# 11 — Hybrid Fusion (Method 10): handcrafted + ABCD + deep + SMOTE + MLP/XGBoost

This is the **Bansal 2022 recipe** adapted to our lesion-cropped data. Concretely:

1. **Handcrafted features** — HOG + HSV color histogram + GLCM (same as the
   classical-ML pipeline in notebook 02), computed on the 128×128 downsample
   of the lesion-cropped 448 image.
2. **ABCD clinical features** — asymmetry (PCA-aligned XOR), border
   irregularity (perimeter² / 4π·area), color diversity (per-channel std in
   RGB+HSV+Lab + L-channel entropy), diameter (fitted-ellipse major axis /
   image diagonal). Mask is recomputed via Otsu on each cropped image at
   evaluation time (no extra storage required).
3. **Deep features** — penultimate layer of one or more trained CNNs. The
   notebook prefers ResNet50 by default but auto-falls back to any CNN
   checkpoint present in `MyDrive/melanoma/checkpoints/`. If multiple
   checkpoints are present, their features are concatenated
   (Bansal-style multi-CNN fusion).
4. **Scaling + PCA(500)** — standardise then reduce; PCA fit on train only.
5. **SMOTE-Tomek** — synthetic minority oversampling + Tomek-link cleanup
   on the TRAIN feature matrix only. Matches the SMOTE-Tomek protocol used
   by Naeem 2024 (SNC_Net), which reached F1 0.981 on ISIC 2019.
6. **Shallow classifier** — train an MLP (Bansal style) and an XGBoost
   classifier (Naeem style) on the balanced training features; pick the
   one with higher val F1.
7. **Threshold tuning + test evaluation** — sweep thresholds on val,
   apply the F1-maximising threshold once on test; save the standard
   `hybrid_fusion_*` outputs so the aggregation notebook ingests this
   as a new row in the comparison table.

**Expected output:** test F1 in the 0.80-0.90 band on HAM10000 binary
with the lesion-grouped split. Bansal 2022 (same dataset, same binary
task, similar recipe but image-grouped split) reports 94.9% acc.

This notebook requires `imbalanced-learn` (auto-installed below) and at
least one CNN checkpoint. **CPU-only run**, around 10–20 minutes.

In [ ]:
# --- Colab setup: ensure the project is on sys.path, mount Drive, load config ---
import os, sys, subprocess
from pathlib import Path

# Either the project is already on disk (uploaded zip / mounted Drive) or we
# clone it from GitHub. We never destroy local changes.
REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Mount Drive (silently re-uses an existing mount on re-run)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Drive root  :", config.DRIVE_ROOT)
print("Data dir    :", config.DATA_DIR)
print("Results dir :", config.RESULTS_DIR)

In [ ]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [ ]:
# Install only what Colab doesn't already ship with.
!pip install --quiet imbalanced-learn xgboost timm 2>&1 | tail -n 1

In [ ]:
# --- Inspect which CNN checkpoints exist on Drive ---
import time
from pathlib import Path

# /content/local_data is reused for intermediate feature caches so a
# disconnect in the middle of this notebook doesn't waste the slow CPU work.
CACHE_DIR = Path("/content/local_data")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ARCHES = ["resnet50", "densenet121", "efficientnet_b3", "swin_tiny", "vgg16_bn", "alexnet"]
available = [a for a in ARCHES
             if (config.CHECKPOINT_DIR / f"{a}_best.pt").exists()]
if not available:
    raise RuntimeError(
        "No CNN checkpoints found in MyDrive/melanoma/checkpoints/. "
        "Run at least one of notebooks 04-09 before this hybrid fusion notebook."
    )
print("Checkpoints available for deep-feature extraction:")
for a in available:
    p = config.CHECKPOINT_DIR / f"{a}_best.pt"
    size_mb = p.stat().st_size / 1e6
    print(f"  - {a:18s}  ({size_mb:.1f} MB)")

# Use EVERY available CNN's penultimate features concatenated — this is
# the Bansal 2022 multi-CNN fusion recipe (ResNet50V2 + EfficientNet-B0
# + handcrafted -> ANN, 94.9% on HAM10000). The single-ResNet50 default
# we shipped first gave a 2048-D deep block; the all-CNN default below
# stacks up to ~13.5K dims (AlexNet 4096 + VGG 4096 + ResNet50 2048 +
# EffNetB3 1536 + DenseNet 1024 + Swin 768) which PCA then projects to
# 500 components, exactly matching the published recipe.
DEEP_FEATURE_SOURCES = available
print(f"\nWill extract deep features from: {DEEP_FEATURE_SOURCES}")

In [ ]:
# --- Load arrays + indices ---
import numpy as np
from src.data import load_arrays

t0 = time.time()
X, y, ids, idx_train, idx_val, idx_test = load_arrays(config.DATA_DIR)
print(f"Loaded arrays in {time.time()-t0:.1f}s.  "
      f"X: {X.shape}  splits: {len(idx_train)}/{len(idx_val)}/{len(idx_test)}")

In [ ]:
# --- Handcrafted features (HOG + Color hist + GLCM) on 128x128 downsample ---
# Parallelised with joblib (A100 runtime has ~12 vCPUs); cached so a
# re-run after interruption skips this slow step.
import cv2
from joblib import Parallel, delayed
from tqdm import tqdm
from src.features import extract_all

F_hand_path = CACHE_DIR / "F_hand.npy"
if F_hand_path.exists():
    F_hand = np.load(F_hand_path)
    print(f"Loaded cached F_hand: {F_hand.shape}  (delete {F_hand_path} to recompute)")
else:
    t0 = time.time()
    # Resize first (cheap), then extract features in parallel.
    print("Resizing 10,015 images to 128x128 ...")
    X128 = np.empty((len(X), config.HOG_IMG_SIZE, config.HOG_IMG_SIZE, 3), dtype=np.uint8)
    for i in range(len(X)):
        X128[i] = cv2.resize(X[i], (config.HOG_IMG_SIZE, config.HOG_IMG_SIZE),
                              interpolation=cv2.INTER_AREA)

    # Bind hyperparameters to local variables so joblib workers receive them
    # without dragging the whole config module across the pickle boundary.
    PPC = config.HOG_PIXELS_PER_CELL
    CPB = config.HOG_CELLS_PER_BLOCK
    CHB = config.COLOR_HIST_BINS
    GD = config.GLCM_DISTANCES
    GA = config.GLCM_ANGLES

    sample = extract_all(X128[0], pixels_per_cell=PPC, cells_per_block=CPB,
                         color_bins=CHB, glcm_distances=GD, glcm_angles=GA)
    print(f"Handcrafted feature dim per image: {sample.shape[0]}  "
          f"(HOG + Color + GLCM)")
    print("Computing handcrafted features in parallel (threading backend) ...")

    feats_list = Parallel(n_jobs=-1, verbose=5, backend="threading", batch_size=128)(
        delayed(extract_all)(X128[i],
                             pixels_per_cell=PPC,
                             cells_per_block=CPB,
                             color_bins=CHB,
                             glcm_distances=GD,
                             glcm_angles=GA)
        for i in range(len(X128))
    )
    F_hand = np.asarray(feats_list, dtype=np.float32)
    del X128, feats_list  # free RAM

    np.save(F_hand_path, F_hand)
    print(f"F_hand: {F_hand.shape}  (computed in {time.time()-t0:.1f}s, "
          f"cached to {F_hand_path})")

In [ ]:
# --- ABCD clinical features (Otsu mask recomputed per cropped image) ---
# Also parallelised + cached.
import cv2
from joblib import Parallel, delayed
from src.abcd import abcd_features, ABCD_FEATURE_NAMES
from src.segmentation import segment_lesion

F_abcd_path = CACHE_DIR / "F_abcd.npy"
if F_abcd_path.exists():
    F_abcd = np.load(F_abcd_path)
    print(f"Loaded cached F_abcd: {F_abcd.shape}  (delete {F_abcd_path} to recompute)")
else:
    t0 = time.time()
    from src.abcd import abcd_features_from_image
    print(f"ABCD feature names ({len(ABCD_FEATURE_NAMES)}): {ABCD_FEATURE_NAMES}")
    print(f"Computing ABCD features in parallel (threading) on {len(X)} images ...")

    results = Parallel(n_jobs=-1, verbose=5, backend="threading", batch_size=128)(
        delayed(abcd_features_from_image)(X[i]) for i in range(len(X))
    )
    F_abcd = np.asarray([r[0] for r in results], dtype=np.float32)
    n_mask_fail = sum(1 for r in results if r[1])
    np.save(F_abcd_path, F_abcd)
    print(f"F_abcd: {F_abcd.shape}  (Otsu fallback in {n_mask_fail} crops, "
          f"computed in {time.time()-t0:.1f}s, cached to {F_abcd_path})")

In [ ]:
# --- Deep features: penultimate layer of each available CNN ---
# Cached so a re-run after interruption skips the GPU pass.
import torch
from torch.utils.data import DataLoader
from src.data import HAMDataset, make_eval_transform
from src.models import BUILDERS, chop_head_for_features, PENULTIMATE_DIMS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device for deep-feature extraction: {device}")

@torch.no_grad()
def extract_deep_features(arch):
    cache_path = CACHE_DIR / f"F_deep_{arch}.npy"
    if cache_path.exists():
        feats = np.load(cache_path)
        print(f"  loaded cached {arch}: {feats.shape}")
        return feats
    t0 = time.time()
    cfg = config.ARCH_CONFIG[arch]
    model = BUILDERS[arch](num_classes=2, pretrained=False).to(device)
    ckpt = torch.load(config.CHECKPOINT_DIR / f"{arch}_best.pt", map_location=device)
    model.load_state_dict(ckpt, strict=False)
    model = chop_head_for_features(model, arch).to(device).eval()
    eval_tf = make_eval_transform(cfg["input_size"])
    ds_all = HAMDataset(X, y, np.arange(len(X)), eval_tf)
    ld = DataLoader(ds_all, batch_size=cfg["batch_size"], shuffle=False,
                    num_workers=2, pin_memory=True)
    feats = np.empty((len(X), PENULTIMATE_DIMS[arch]), dtype=np.float32)
    off = 0
    for x_batch, _ in tqdm(ld, desc=f"deep features [{arch}]"):
        x_batch = x_batch.to(device, non_blocking=True)
        out = model(x_batch)
        if out.dim() > 2:
            out = out.flatten(start_dim=1)
        n = out.shape[0]
        feats[off:off + n] = out.cpu().numpy()
        off += n
    np.save(cache_path, feats)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"  {arch}: {feats.shape}  (extracted in {time.time()-t0:.1f}s, "
          f"cached to {cache_path})")
    return feats

F_deep_parts = []
deep_names = []
for arch in DEEP_FEATURE_SOURCES:
    feats = extract_deep_features(arch)
    F_deep_parts.append(feats)
    deep_names.append(f"{arch}({PENULTIMATE_DIMS[arch]})")
F_deep = np.concatenate(F_deep_parts, axis=1)
print(f"F_deep ({' + '.join(deep_names)}): {F_deep.shape}")

In [ ]:
# --- Concatenate all feature blocks ---
F_all = np.concatenate([F_hand, F_abcd, F_deep], axis=1).astype(np.float32)
print(f"F_all: {F_all.shape}  "
      f"(hand={F_hand.shape[1]}, abcd={F_abcd.shape[1]}, deep={F_deep.shape[1]})  "
      f"size in RAM: {F_all.nbytes / 1e9:.2f} GB")

In [ ]:
# --- StandardScaler (no centering) + TruncatedSVD, fit on TRAIN only ---
# Why TruncatedSVD instead of PCA:
#   With the Bansal multi-CNN deep block, F_all is (~10K, ~16K) — too
#   wide for sklearn's randomized-PCA path on Colab CPU (hangs > 8 min).
#   TruncatedSVD skips the zero-centering step that makes the matrix
#   dense and slow; it finishes in ~30-90s on the same shape with
#   negligible quality loss for the downstream MLP/XGBoost.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD

t0 = time.time()
print("Fitting StandardScaler (with_mean=False) on train ...")
scaler = StandardScaler(with_mean=False).fit(F_all[idx_train])
F_scaled = scaler.transform(F_all).astype(np.float32)
print(f"  scaler done in {time.time()-t0:.1f}s.  F_scaled: {F_scaled.shape}")

# Cap the component count at min(300, n_features-1, n_train-1).
# 300 is enough to capture the variance once the multi-CNN deep block
# stacks 13.5K features through PCA — a 500-component target was the
# old default that triggered the hang.
n_comp = int(min(300, F_scaled.shape[1] - 1, len(idx_train) - 1))
t0 = time.time()
print(f"Fitting TruncatedSVD(n={n_comp}, n_iter=5) on train ...")
tsvd = TruncatedSVD(
    n_components=n_comp,
    n_iter=5,
    random_state=config.SEED,
).fit(F_scaled[idx_train])
F_pca = tsvd.transform(F_scaled).astype(np.float32)
print(f"TruncatedSVD done in {time.time()-t0:.1f}s.  kept {n_comp} components, "
      f"explained variance = {tsvd.explained_variance_ratio_.sum():.3f}")
print(f"F_pca: {F_pca.shape}")

In [ ]:
# --- SMOTE-Tomek on TRAIN ONLY ---
# (val/test must remain at the natural 1:8 distribution for honest evaluation.)
from imblearn.combine import SMOTETomek

t0 = time.time()
Xtr_pca = F_pca[idx_train]
ytr     = y[idx_train]
print(f"Before SMOTE-Tomek: train={Xtr_pca.shape}, "
      f"class counts={dict(zip(*np.unique(ytr, return_counts=True)))}")
print("Running SMOTE-Tomek ...")

smt = SMOTETomek(random_state=config.SEED, n_jobs=-1)
Xtr_bal, ytr_bal = smt.fit_resample(Xtr_pca, ytr)
print(f"After  SMOTE-Tomek: train={Xtr_bal.shape}, "
      f"class counts={dict(zip(*np.unique(ytr_bal, return_counts=True)))}  "
      f"({time.time()-t0:.1f}s)")

Xv_pca = F_pca[idx_val];  yv = y[idx_val]
Xt_pca = F_pca[idx_test]; yt = y[idx_test]

In [ ]:
# --- Train MLP (Bansal-style ANN) ---
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score

print("Training MLP [256 -> 128] with early stopping ...")
t0 = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=config.SEED,
    verbose=False,
)
mlp.fit(Xtr_bal, ytr_bal)
mlp_train_time = time.time() - t0

pv_mlp = mlp.predict_proba(Xv_pca)[:, 1]
pt_mlp = mlp.predict_proba(Xt_pca)[:, 1]
print(f"MLP trained in {mlp_train_time:.1f}s ({mlp.n_iter_} iters).  "
      f"val AUC = {roc_auc_score(yv, pv_mlp):.4f}")

In [ ]:
# --- Train XGBoost (Naeem-style booster) ---
import xgboost as xgb

print("Training XGBoost (500 trees, depth=6, hist) ...")
t0 = time.time()
xgbc = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=config.SEED,
    n_jobs=-1,
    tree_method="hist",
)
xgbc.fit(Xtr_bal, ytr_bal,
         eval_set=[(Xv_pca, yv)],
         verbose=False)
xgb_train_time = time.time() - t0

pv_xgb = xgbc.predict_proba(Xv_pca)[:, 1]
pt_xgb = xgbc.predict_proba(Xt_pca)[:, 1]
print(f"XGBoost trained in {xgb_train_time:.1f}s.  val AUC = {roc_auc_score(yv, pv_xgb):.4f}")

In [ ]:
# --- Pick the better classifier on val F1 (at tuned threshold) ---
from src.training import tune_threshold

best_t_mlp, val_f1_mlp = tune_threshold(yv, pv_mlp)
best_t_xgb, val_f1_xgb = tune_threshold(yv, pv_xgb)
print(f"MLP    : val F1 = {val_f1_mlp:.4f} at t = {best_t_mlp:.3f}")
print(f"XGBoost: val F1 = {val_f1_xgb:.4f} at t = {best_t_xgb:.3f}")

if val_f1_mlp >= val_f1_xgb:
    chosen, pv, pt, best_t = "MLP", pv_mlp, pt_mlp, best_t_mlp
    train_time = mlp_train_time
else:
    chosen, pv, pt, best_t = "XGBoost", pv_xgb, pt_xgb, best_t_xgb
    train_time = xgb_train_time

print(f"\nChosen classifier: {chosen}  (val F1 = {max(val_f1_mlp, val_f1_xgb):.4f})")

In [ ]:
# --- Final test evaluation with the chosen classifier + tuned threshold ---
import time
from src.evaluation import save_standard_outputs, compute_metrics

t0 = time.time()
yp_test = (pt > best_t).astype(int)
inf_ms_per_image = (time.time() - t0) * 1000.0 / max(1, len(pt))

hp = dict(
    classifier=chosen,
    deep_feature_sources=list(DEEP_FEATURE_SOURCES),
    handcrafted_dim=int(F_hand.shape[1]),
    abcd_dim=int(F_abcd.shape[1]),
    deep_dim=int(F_deep.shape[1]),
    pca_components=int(n_comp),
    pca_explained_variance=float(tsvd.explained_variance_ratio_.sum()),
    balancing="SMOTE-Tomek (train only)",
    train_class_counts_before=dict(zip(*[a.tolist() for a in np.unique(ytr, return_counts=True)])),
    train_class_counts_after=dict(zip(*[a.tolist() for a in np.unique(ytr_bal, return_counts=True)])),
    decision_threshold=float(best_t),
    threshold_selection="argmax F1 on validation set",
    val_f1_mlp=float(val_f1_mlp),
    val_f1_xgb=float(val_f1_xgb),
    notes="Bansal2022 + Naeem2024 inspired hybrid fusion (handcrafted + ABCD + deep features).",
)

metrics = save_standard_outputs(
    method_name="hybrid_fusion",
    results_dir=config.RESULTS_DIR,
    y_true=yt,
    y_pred=yp_test,
    y_prob=pt,
    ids=ids[idx_test],
    hyperparameters=hp,
    train_time_sec=float(train_time),
    inference_time_per_image_ms=float(inf_ms_per_image),
)

# Save val probs too — useful for later soft-voting against the CNN ensemble.
import pandas as pd
pd.DataFrame({
    "image_id": ids[idx_val], "y_true": yv,
    "y_prob_mlp": pv_mlp, "y_prob_xgb": pv_xgb,
    "y_prob_chosen": pv, "best_t": best_t,
}).to_csv(config.RESULTS_DIR / "hybrid_fusion_val_predictions.csv", index=False)

print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, (int, float))})

---

## What this produces

- `results/hybrid_fusion_metrics.json` — headline metrics (acc/prec/recall/F1/AUC).
- `results/hybrid_fusion_predictions.csv` — per-test-image probabilities and the
  tuned-threshold prediction.
- `results/hybrid_fusion_val_predictions.csv` — per-val-image probabilities from
  both MLP and XGBoost, so you can also try a CNN-ensemble + hybrid soft-vote
  if you want one more row.
- `results/hybrid_fusion_confusion_matrix.png`, `_roc_curve.png`.

After this notebook finishes, re-run **`10_aggregation.ipynb`** — it auto-detects
the new `hybrid_fusion` entry and appends it to the comparison table. The IEEE
paper draft (`paper/paper_draft.tex`) then gets a 10th row.

## If you also want a hybrid + CNN-ensemble super-fusion

After both 10_aggregation and this notebook have finished, you can compute an
additional soft-vote between `ensemble` (6 CNNs) and `hybrid_fusion` (this
notebook) by averaging the two test probabilities at their respective tuned
thresholds. That's a single extra cell.